In [1]:
# Cell 1: Setup

import os
import sys

# Install packages
for pkg in ['tqdm', 'joblib', 'rasterio', 'skyfield', 'folium', 'pyproj']:
    os.system(f'{sys.executable} -m pip install -q {pkg}')

# Configuration
TLE_URL = 'https://celestrak.org/NORAD/elements/gp.php?GROUP=globalstar&FORMAT=tle'
MAPS_DIR = 'maps'
os.makedirs(MAPS_DIR, exist_ok=True)

print(f"✓ TLE_URL: {TLE_URL}")
print(f"✓ MAPS_DIR: {MAPS_DIR}")

✓ TLE_URL: https://celestrak.org/NORAD/elements/gp.php?GROUP=globalstar&FORMAT=tle
✓ MAPS_DIR: maps


In [9]:
# Cell 2: Define Analysis Area (Linked to Auto-Config)

import rasterio
from pyproj import Transformer
import os
import numpy as np

print("="*70)
print("DEFINING ANALYSIS AREA")
print("="*70)

# --- 1. Check for Dynamic File Path ---
# We check if Cell 2-AUTO has already defined the file path
if 'DEM_FILE_PATH' not in globals():
    print("❌ ERROR: DEM_FILE_PATH variable is missing.")
    print("   Please run 'Cell 2-AUTO' first to select your DEM file.")
else:
    # --- 2. Verify File Exists ---
    if not os.path.exists(DEM_FILE_PATH):
        print(f"❌ ERROR: File not found at path: {DEM_FILE_PATH}")
        print("   Please check that the file exists in your data folder.")
    else:
        print(f"✓ Using DEM: {DEM_FILE_PATH}")
        
        # --- 3. Auto-extract bounds from DEM ---
        with rasterio.open(DEM_FILE_PATH) as dem:
            bounds = dem.bounds
            
            # Create a transformer to convert from the DEM's native coordinate system
            # (likely UTM) to standard Latitude/Longitude (EPSG:4326)
            transformer = Transformer.from_crs(dem.crs, "EPSG:4326", always_xy=True)
            
            # Transform the corner coordinates
            lon_min, lat_min = transformer.transform(bounds.left, bounds.bottom)
            lon_max, lat_max = transformer.transform(bounds.right, bounds.top)

        # --- 4. Calculate and Print Area Info ---
        center_lat = (lat_min + lat_max) / 2
        center_lon = (lon_min + lon_max) / 2

        print(f"\nAnalysis Area:")
        print(f"  Latitude:  {lat_min:.4f}° to {lat_max:.4f}°N")
        print(f"  Longitude: {lon_min:.4f}° to {lon_max:.4f}°W")
        print(f"  Center:    {center_lat:.4f}°N, {center_lon:.4f}°W")
        
        # Calculate approximate size in km
        size_lat_km = (lat_max-lat_min) * 111.0
        size_lon_km = (lon_max-lon_min) * 111.0 * np.cos(np.radians(center_lat))
        print(f"  Size:      ~{size_lat_km:.1f} x {size_lon_km:.1f} km")

print("="*70)

DEFINING ANALYSIS AREA
✓ Using DEM: C:\Users\caleb\OneDrive\Desktop\SatDEMs\mtBaldy.tif

Analysis Area:
  Latitude:  34.2575° to 34.2970°N
  Longitude: -117.6552° to -117.5935°W
  Center:    34.2773°N, -117.6243°W
  Size:      ~4.4 x 5.7 km


In [10]:
# Cell 2: Satellite Loading Functions

import requests
from skyfield.api import EarthSatellite, load

def fetch_globalstar_satellites(ts):
    """
    Fetch all Globalstar satellites from Celestrak.
    
    Args:
        ts: Skyfield timescale object
    
    Returns:
        List of EarthSatellite objects
    """
    url = "https://celestrak.org/NORAD/elements/gp.php?GROUP=globalstar&FORMAT=tle"
    
    try:
        print(f"Fetching Globalstar TLEs from Celestrak...")
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        
        lines = [line.strip() for line in response.text.strip().split('\n') if line.strip()]
        
        satellites = []
        i = 0
        while i < len(lines):
            if i + 2 < len(lines):
                name = lines[i]
                line1 = lines[i + 1]
                line2 = lines[i + 2]
                
                # Verify TLE format
                if line1.startswith('1 ') and line2.startswith('2 '):
                    sat = EarthSatellite(line1, line2, name, ts)
                    satellites.append(sat)
                    i += 3
                else:
                    i += 1
            else:
                break
        
        print(f"✓ Loaded {len(satellites)} Globalstar satellites")
        return satellites
        
    except Exception as e:
        print(f"✗ Error fetching satellites: {e}")
        return []

def print_satellite_summary(satellites):
    """Print summary of loaded satellites."""
    print(f"\n{'='*60}")
    print(f"GLOBALSTAR SATELLITE FLEET")
    print(f"{'='*60}")
    print(f"Total satellites: {len(satellites)}")
    print(f"\nSatellites loaded:")
    for i, sat in enumerate(satellites, 1):
        print(f"  {i:2d}. {sat.name}")
    print(f"{'='*60}\n")

print("✓ Satellite loading functions defined")

✓ Satellite loading functions defined


In [11]:
# Cell 2-AUTO: Automatically extract bounds from DEM file (Non-Rectangular Support)

import os
import numpy as np
import rasterio
from pyproj import Transformer

print("="*70)
print("SATELLITE LINE-OF-SIGHT ANALYSIS - AUTO CONFIGURATION")
print("="*70)

# ========== USER INPUT ==========
DEM_FILENAME = input("\nEnter DEM filename (e.g., 'woods_canyon.tif'): ").strip()

if not DEM_FILENAME:
    raise ValueError("DEM filename cannot be empty!")

# Add .tif extension if not provided
if not DEM_FILENAME.lower().endswith(('.tif', '.tiff')):
    print(f"  Note: Adding .tif extension")
    DEM_FILENAME = DEM_FILENAME + '.tif'

print(f"\n✓ Using DEM file: {DEM_FILENAME}")
# ================================

# Set up directories
BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, 'data', 'dem')
OUTPUT_DIR = os.path.join(BASE_DIR, 'output')
KML_DIR = os.path.join(OUTPUT_DIR, 'kml')
MAPS_DIR = os.path.join(OUTPUT_DIR, 'maps')

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(KML_DIR, exist_ok=True)
os.makedirs(MAPS_DIR, exist_ok=True)

DEM_FILE_PATH = os.path.join(DATA_DIR, DEM_FILENAME)

# Verify file exists
if not os.path.exists(DEM_FILE_PATH):
    print(f"\n✗ ERROR: DEM file not found!")
    print(f"  Expected location: {DEM_FILE_PATH}")
    print(f"\n  Please ensure '{DEM_FILENAME}' is in the '{DATA_DIR}' directory")
    
    # List available DEM files
    if os.path.exists(DATA_DIR):
        dem_files = [f for f in os.listdir(DATA_DIR) if f.lower().endswith(('.tif', '.tiff'))]
        if dem_files:
            print(f"\n  Available DEM files in {DATA_DIR}:")
            for f in dem_files:
                file_size = os.path.getsize(os.path.join(DATA_DIR, f)) / (1024*1024)
                print(f"    - {f} ({file_size:.1f} MB)")
        else:
            print(f"\n  No .tif files found in {DATA_DIR}")
    
    raise FileNotFoundError(f"DEM file not found: {DEM_FILE_PATH}")

file_size_mb = os.path.getsize(DEM_FILE_PATH) / (1024*1024)
print(f"✓ DEM file found: {DEM_FILENAME} ({file_size_mb:.1f} MB)")

# Open DEM and extract bounds
print("\nAnalyzing DEM structure...")
with rasterio.open(DEM_FILE_PATH) as dem:
    dem_crs_temp = dem.crs
    dem_bounds_native = dem.bounds
    dem_shape = dem.shape
    dem_nodata = dem.nodata
    
    print(f"  Dimensions: {dem_shape[0]} rows × {dem_shape[1]} columns")
    print(f"  CRS: {dem_crs_temp}")
    print(f"  NoData value: {dem_nodata}")
    
    # Read a sample of the data to check for valid pixels
    print("\nExtracting geographic bounds...")
    
    # Method 1: Simple bounding box (works for rectangular DEMs)
    transformer = Transformer.from_crs(dem_crs_temp, "EPSG:4326", always_xy=True)
    
    # Transform all four corners to handle rotation/skew
    corners_native = [
        (dem_bounds_native.left, dem_bounds_native.bottom),   # SW
        (dem_bounds_native.right, dem_bounds_native.bottom),  # SE
        (dem_bounds_native.right, dem_bounds_native.top),     # NE
        (dem_bounds_native.left, dem_bounds_native.top),      # NW
    ]
    
    corners_latlon = []
    for x, y in corners_native:
        lon, lat = transformer.transform(x, y)
        corners_latlon.append((lat, lon))
    
    # Extract min/max from all corners (handles rotated DEMs)
    lats = [c[0] for c in corners_latlon]
    lons = [c[1] for c in corners_latlon]
    
    lon_min_simple = min(lons)
    lon_max_simple = max(lons)
    lat_min_simple = min(lats)
    lat_max_simple = max(lats)
    
    # Method 2: Sample-based bounds (for non-rectangular/masked DEMs)
    print("  Checking for non-rectangular geometry...")
    
    # Read full DEM data
    dem_data_temp = dem.read(1)
    
    # Identify valid (non-nodata) pixels
    if dem_nodata is not None:
        valid_mask = dem_data_temp != dem_nodata
    else:
        # If no nodata value specified, check for common nodata values
        valid_mask = ~np.isnan(dem_data_temp)
        valid_mask &= (dem_data_temp != -9999)
        valid_mask &= (dem_data_temp != -3.4028235e+38)  # Common float nodata
    
    valid_pixels = np.sum(valid_mask)
    total_pixels = dem_data_temp.size
    valid_percentage = (valid_pixels / total_pixels) * 100
    
    print(f"  Valid pixels: {valid_pixels:,} / {total_pixels:,} ({valid_percentage:.1f}%)")
    
    # If less than 95% valid, use sample-based approach
    is_non_rectangular = valid_percentage < 95.0
    
    if is_non_rectangular:
        print("  ⚠ Non-rectangular DEM detected - using precise bounds calculation")
        
        # Sample valid pixels to find actual data extent
        valid_rows, valid_cols = np.where(valid_mask)
        
        if len(valid_rows) == 0:
            raise ValueError("No valid data found in DEM!")
        
        # Get bounds of valid data in pixel coordinates
        row_min, row_max = valid_rows.min(), valid_rows.max()
        col_min, col_max = valid_cols.min(), valid_cols.max()
        
        # Sample points along the edges of valid data
        sample_points = []
        
        # Top edge
        for col in range(col_min, col_max + 1, max(1, (col_max - col_min) // 20)):
            rows_at_col = valid_rows[valid_cols == col]
            if len(rows_at_col) > 0:
                sample_points.append((rows_at_col.min(), col))
        
        # Bottom edge
        for col in range(col_min, col_max + 1, max(1, (col_max - col_min) // 20)):
            rows_at_col = valid_rows[valid_cols == col]
            if len(rows_at_col) > 0:
                sample_points.append((rows_at_col.max(), col))
        
        # Left edge
        for row in range(row_min, row_max + 1, max(1, (row_max - row_min) // 20)):
            cols_at_row = valid_cols[valid_rows == row]
            if len(cols_at_row) > 0:
                sample_points.append((row, cols_at_row.min()))
        
        # Right edge
        for row in range(row_min, row_max + 1, max(1, (row_max - row_min) // 20)):
            cols_at_row = valid_cols[valid_rows == row]
            if len(cols_at_row) > 0:
                sample_points.append((row, cols_at_row.max()))
        
        # Convert sample points to lat/lon
        sample_lats = []
        sample_lons = []
        
        for row, col in sample_points:
            x, y = dem.transform * (col, row)
            lon, lat = transformer.transform(x, y)
            sample_lats.append(lat)
            sample_lons.append(lon)
        
        lon_min = min(sample_lons)
        lon_max = max(sample_lons)
        lat_min = min(sample_lats)
        lat_max = max(sample_lats)
        
        print(f"  Sampled {len(sample_points)} edge points for precise bounds")
        
    else:
        print("  ✓ Rectangular DEM - using standard bounding box")
        lon_min = lon_min_simple
        lon_max = lon_max_simple
        lat_min = lat_min_simple
        lat_max = lat_max_simple

# Calculate center
center_lat = (lat_min + lat_max) / 2
center_lon = (lon_min + lon_max) / 2

# Store boundary corners (for rectangular approximation)
BOUNDARY_CORNERS = [
    (lat_max, lon_min),  # NW
    (lat_max, lon_max),  # NE
    (lat_min, lon_max),  # SE
    (lat_min, lon_min),  # SW
]

# Calculate area
area_km_lat = abs(lat_max - lat_min) * 111
area_km_lon = abs(lon_max - lon_min) * 111 * np.cos(np.radians(center_lat))

# Display configuration
print("\n" + "="*70)
print("CONFIGURATION SUMMARY (AUTO-EXTRACTED)")
print("="*70)
print(f"DEM File: {DEM_FILENAME}")
print(f"Location: {DATA_DIR}")
print(f"File Size: {file_size_mb:.1f} MB")

if is_non_rectangular:
    print(f"\nDEM Type: Non-Rectangular (Masked/Irregular)")
    print(f"  Valid data coverage: {valid_percentage:.1f}%")
else:
    print(f"\nDEM Type: Rectangular")

print(f"\nAnalysis Area (from valid DEM data):")
print(f"  Center: {center_lat:.6f}°N, {center_lon:.6f}°W")
print(f"  Latitude:  {lat_min:.6f}° to {lat_max:.6f}°N")
print(f"  Longitude: {lon_min:.6f}° to {lon_max:.6f}°W")
print(f"  Dimensions: ~{area_km_lat:.2f} km (N-S) × {area_km_lon:.2f} km (E-W)")
print(f"  Total Area: ~{area_km_lat * area_km_lon:.2f} km²")

# Show corner coordinates
print(f"\nBounding Box Corners:")
print(f"  NW: {lat_max:.6f}°N, {lon_min:.6f}°W")
print(f"  NE: {lat_max:.6f}°N, {lon_max:.6f}°W")
print(f"  SE: {lat_min:.6f}°N, {lon_max:.6f}°W")
print(f"  SW: {lat_min:.6f}°N, {lon_min:.6f}°W")

print("="*70)
print("\n✓ Configuration complete! Bounds automatically extracted from DEM.")

if is_non_rectangular:
    print("\n📝 Note: Non-rectangular DEM detected.")
    print("   LOS analysis will only work within valid data regions.")

print("\n📍 Next Step: Run Cell 3 to continue with satellite configuration")

SATELLITE LINE-OF-SIGHT ANALYSIS - AUTO CONFIGURATION



Enter DEM filename (e.g., 'woods_canyon.tif'):  yosemiteValley.tif



✓ Using DEM file: yosemiteValley.tif
✓ DEM file found: yosemiteValley.tif (288.5 MB)

Analyzing DEM structure...
  Dimensions: 9119 rows × 16638 columns
  CRS: EPSG:26911
  NoData value: -9999.0

Extracting geographic bounds...
  Checking for non-rectangular geometry...
  Valid pixels: 94,127,086 / 151,721,922 (62.0%)
  ⚠ Non-rectangular DEM detected - using precise bounds calculation
  Sampled 84 edge points for precise bounds

CONFIGURATION SUMMARY (AUTO-EXTRACTED)
DEM File: yosemiteValley.tif
Location: C:\Users\caleb\anaconda_projects\satellite-los-analysis\data\dem
File Size: 288.5 MB

DEM Type: Non-Rectangular (Masked/Irregular)
  Valid data coverage: 62.0%

Analysis Area (from valid DEM data):
  Center: 37.734034°N, -119.601210°W
  Latitude:  37.691686° to 37.776382°N
  Longitude: -119.695063° to -119.507357°W
  Dimensions: ~9.40 km (N-S) × 16.48 km (E-W)
  Total Area: ~154.91 km²

Bounding Box Corners:
  NW: 37.776382°N, -119.695063°W
  NE: 37.776382°N, -119.507357°W
  SE: 37.6

In [12]:
# Cell 3: Initialize Skyfield and fetch Globalstar satellites
print("\n" + "="*60)
print("INITIALIZING SATELLITE DATA")
print("="*60)

ts = load.timescale()
print("✓ Skyfield timescale initialized")

globalstar_fleet = fetch_globalstar_satellites(ts)
print_satellite_summary(globalstar_fleet)

NUM_SATELLITES = len(globalstar_fleet)
print(f"✓ Ready to analyze {NUM_SATELLITES} satellites")


INITIALIZING SATELLITE DATA
✓ Skyfield timescale initialized
Fetching Globalstar TLEs from Celestrak...
✓ Loaded 85 Globalstar satellites

GLOBALSTAR SATELLITE FLEET
Total satellites: 85

Satellites loaded:
   1. GLOBALSTAR M001
   2. GLOBALSTAR M004
   3. GLOBALSTAR M002
   4. GLOBALSTAR M003
   5. GLOBALSTAR M014
   6. GLOBALSTAR M006
   7. GLOBALSTAR M015
   8. GLOBALSTAR M008
   9. GLOBALSTAR M023
  10. GLOBALSTAR M040
  11. GLOBALSTAR M036
  12. GLOBALSTAR M038
  13. GLOBALSTAR M022
  14. GLOBALSTAR M041
  15. GLOBALSTAR M046
  16. GLOBALSTAR M037
  17. GLOBALSTAR M045
  18. GLOBALSTAR M019
  19. GLOBALSTAR M044
  20. GLOBALSTAR M042
  21. GLOBALSTAR M025
  22. GLOBALSTAR M049
  23. GLOBALSTAR M047
  24. GLOBALSTAR M052
  25. GLOBALSTAR M035
  26. GLOBALSTAR M032
  27. GLOBALSTAR M051
  28. GLOBALSTAR M030
  29. GLOBALSTAR M048
  30. GLOBALSTAR M026
  31. GLOBALSTAR M043
  32. GLOBALSTAR M028
  33. GLOBALSTAR M024
  34. GLOBALSTAR M027
  35. GLOBALSTAR M054
  36. GLOBALSTAR M053


In [13]:
# Cell 4: Load Satellites (M073 and up only)

from skyfield.api import load

print("Loading satellites...")
satellites = load.tle_file(TLE_URL)

# Filter for Globalstar M073 and up
globalstar_fleet = []
for sat in satellites:
    if 'GLOBALSTAR M' in sat.name.upper():
        # Extract number (e.g., "GLOBALSTAR M082" -> 82)
        try:
            num = int(sat.name.split('M')[-1])
            if num >= 73:
                globalstar_fleet.append(sat)
        except:
            pass

satellites = globalstar_fleet  # Replace satellites list

print(f"✓ Loaded {len(satellites)} Globalstar satellites (M073+)")
for sat in satellites[:25]:
    print(f"  {sat.name}")

Loading satellites...
✓ Loaded 25 Globalstar satellites (M073+)
  GLOBALSTAR M079
  GLOBALSTAR M074
  GLOBALSTAR M076
  GLOBALSTAR M077
  GLOBALSTAR M075
  GLOBALSTAR M073
  GLOBALSTAR M083
  GLOBALSTAR M088
  GLOBALSTAR M091
  GLOBALSTAR M085
  GLOBALSTAR M081
  GLOBALSTAR M089
  GLOBALSTAR M084
  GLOBALSTAR M080
  GLOBALSTAR M082
  GLOBALSTAR M092
  GLOBALSTAR M090
  GLOBALSTAR M086
  GLOBALSTAR M097
  GLOBALSTAR M093
  GLOBALSTAR M094
  GLOBALSTAR M096
  GLOBALSTAR M078
  GLOBALSTAR M095
  GLOBALSTAR M087


In [14]:
# Cell 5: Set analysis time using Skyfield

from skyfield.api import load
from datetime import datetime, timezone

# Load timescale
ts = load.timescale()

# Use current time
analysis_time = ts.now()

print("="*70)
print("ANALYSIS TIME CONFIGURATION")
print("="*70)
print(f"Analysis time: {analysis_time.utc_datetime()}")
print(f"UTC ISO format: {analysis_time.utc_iso()}")
print("="*70)

ANALYSIS TIME CONFIGURATION
Analysis time: 2025-11-18 01:51:52.181169+00:00
UTC ISO format: 2025-11-18T01:51:52Z


In [15]:
# Cell 5a: Define Observer Location (Auto-Centered)
# RUN THIS CELL BEFORE RUNNING YOUR CELL 6

import rasterio
from pyproj import Transformer
import os
import numpy as np

print("="*70)
print("DEFINING OBSERVER LOCATION")
print("="*70)

# --- 1. Define File Path (Dynamic) ---
if 'DEM_FILE_PATH' not in globals():
    print("❌ ERROR: DEM_FILE_PATH is not defined.")
    print("   Please run Cell 2-AUTO first.")
elif not os.path.exists(DEM_FILE_PATH):
    print(f"❌ ERROR: File not found at: {DEM_FILE_PATH}")
else:
    print(f"✓ Using DEM: {DEM_FILE_PATH}")

    # --- 2. Helper function to get elevation ---
    def get_elevation_at_point(dem_dataset, lon, lat, dem_crs, wgs84_crs):
        """
        Transforms a Lat/Lon point to the DEM's CRS and samples the elevation.
        """
        # Create a transformer for this specific task
        transformer = Transformer.from_crs(wgs84_crs, dem_crs, always_xy=True)
        
        # Transform coordinates from WGS84 (degrees) to the DEM's CRS (meters)
        x_meter, y_meter = transformer.transform(lon, lat)
        
        # Sample the DEM at the transformed meter-based coordinates
        elevation_gen = dem_dataset.sample([(x_meter, y_meter)])
        
        # Extract and return the elevation value
        try:
            elevation = next(elevation_gen)[0]
            if elevation < -1000: # Check for nodata values
                 return 0.0
            return float(elevation)
        except StopIteration:
            return 0.0 # Point was outside DEM bounds

    # --- 3. Find Center of DEM ---
    with rasterio.open(DEM_FILE_PATH) as dem:
        # Get DEM's CRS (e.g., UTM)
        dem_crs = dem.crs
        # Get WGS84 CRS (Lat/Lon)
        wgs84_crs = "EPSG:4326"
        
        # Create transformer to find the center in Lat/Lon
        bounds_transformer = Transformer.from_crs(dem_crs, wgs84_crs, always_xy=True)
        
        # Get bounds in meters
        bounds = dem.bounds
        
        # Transform corners to Lat/Lon
        lon_min, lat_min = bounds_transformer.transform(bounds.left, bounds.bottom)
        lon_max, lat_max = bounds_transformer.transform(bounds.right, bounds.top)

        # Calculate center in Lat/Lon
        center_lat = (lat_min + lat_max) / 2
        center_lon = (lon_min + lon_max) / 2
        
        # --- 4. Get Elevation at Center ---
        center_elev = get_elevation_at_point(dem, center_lon, center_lat, dem_crs, wgs84_crs)
        
        # --- 5. DEFINE THE CRITICAL VARIABLES ---
        OBSERVER_LAT = center_lat
        OBSERVER_LON = center_lon
        OBSERVER_ALT = center_elev

        print(f"\nObserver Location Set (Map Center):")
        print(f"  Latitude:  {OBSERVER_LAT:.6f}°N")
        print(f"  Longitude: {OBSERVER_LON:.6f}°W")
        print(f"  Elevation: {OBSERVER_ALT:.2f} meters")

print("="*70)

DEFINING OBSERVER LOCATION
✓ Using DEM: C:\Users\caleb\anaconda_projects\satellite-los-analysis\data\dem\yosemiteValley.tif

Observer Location Set (Map Center):
  Latitude:  37.733558°N
  Longitude: -119.600996°W
  Elevation: 1212.80 meters


In [16]:
# Cell 6: Calculate satellite positions relative to observer

from skyfield.api import wgs84, load

print("="*70)
print("CALCULATING SATELLITE POSITIONS")
print("="*70)

# Ensure analysis_time is a Skyfield Time object
if not hasattr(analysis_time, 'utc'):
    ts = load.timescale()
    skyfield_time = ts.from_datetime(analysis_time)
else:
    skyfield_time = analysis_time

print(f"Analysis time: {skyfield_time.utc_datetime()}")
print()

# Create observer location
observer = wgs84.latlon(OBSERVER_LAT, OBSERVER_LON, elevation_m=OBSERVER_ALT)

# Filter for Globalstar satellites
globalstar_fleet = [sat for sat in satellites if 'GLOBALSTAR' in sat.name.upper()]

print(f"Found {len(globalstar_fleet)} Globalstar satellites")
print(f"Calculating positions from observer at {OBSERVER_LAT:.6f}°N, {OBSERVER_LON:.6f}°W")
print()

# Calculate positions for all Globalstar satellites
satellite_positions = {}
visible_satellites = []

for sat in globalstar_fleet:
    # Calculate satellite position
    difference = sat - observer
    topocentric = difference.at(skyfield_time)  # ← FIXED: use skyfield_time
    alt, az, distance = topocentric.altaz()
    
    # Store position data
    satellite_positions[sat.name] = {
        'elevation': alt.degrees,
        'azimuth': az.degrees,
        'distance': distance.km,
        'topocentric': topocentric
    }
    
    # Track visible satellites (above horizon)
    if alt.degrees > 0:
        visible_satellites.append(sat)

# Display results
print(f"Satellites above horizon: {len(visible_satellites)} / {len(globalstar_fleet)}")
print()

# Show visible satellites
if len(visible_satellites) > 0:
    print("Visible Satellites:")
    for idx, sat in enumerate(visible_satellites, 1):
        pos = satellite_positions[sat.name]
        print(f"  [{idx}] {sat.name}")
        print(f"      Elevation: {pos['elevation']:.2f}°")
        print(f"      Azimuth: {pos['azimuth']:.2f}°")
        print(f"      Distance: {pos['distance']:.1f} km")
else:
    print("⚠ No satellites above horizon at this time")
    print("  Try changing the analysis time in Cell 5")

print("="*70)

# At end of Cell 6
visible_satellites = [sat for sat in satellites if satellite_positions[sat.name]['elevation'] > 0]

print(f"\nVisible satellites: {len(visible_satellites)}")

CALCULATING SATELLITE POSITIONS
Analysis time: 2025-11-18 01:51:52.181169+00:00

Found 25 Globalstar satellites
Calculating positions from observer at 37.733558°N, -119.600996°W

Satellites above horizon: 2 / 25

Visible Satellites:
  [1] GLOBALSTAR M084
      Elevation: 30.74°
      Azimuth: 200.67°
      Distance: 2275.0 km
  [2] GLOBALSTAR M082
      Elevation: 17.48°
      Azimuth: 40.16°
      Distance: 2959.8 km

Visible satellites: 2


In [17]:
# Cell 7: Load DEM data and extract CRS
print("Loading DEM data...")

with rasterio.open(DEM_FILE_PATH) as dem:
    dem_data = dem.read(1)
    dem_transform = dem.transform
    dem_crs = dem.crs
    
    # Get actual DEM bounds in its native CRS
    dem_bounds_native = dem.bounds
    
    # Transform DEM corners to lat/lon to verify alignment
    transformer_check = Transformer.from_crs(dem_crs, "EPSG:4326", always_xy=True)
    
    dem_lon_min, dem_lat_min = transformer_check.transform(dem_bounds_native.left, dem_bounds_native.bottom)
    dem_lon_max, dem_lat_max = transformer_check.transform(dem_bounds_native.right, dem_bounds_native.top)

print(f"✓ DEM loaded: {dem_data.shape}")
print(f"  CRS: {dem_crs}")
print(f"  Min elevation: {dem_data.min():.1f}m")
print(f"  Max elevation: {dem_data.max():.1f}m")
print(f"\nDEM Geographic Bounds (from file):")
print(f"  Latitude:  {dem_lat_min:.4f}° to {dem_lat_max:.4f}°N")
print(f"  Longitude: {dem_lon_min:.4f}° to {dem_lon_max:.4f}°W")
print(f"\nUser Input Bounds:")
print(f"  Latitude:  {lat_min:.4f}° to {lat_max:.4f}°N")
print(f"  Longitude: {lon_min:.4f}° to {lon_max:.4f}°W")

# Check if observer is within DEM bounds
if not (dem_lat_min <= center_lat <= dem_lat_max and 
        dem_lon_min <= center_lon <= dem_lon_max):
    print(f"\n⚠ WARNING: Observer location ({center_lat:.4f}°N, {center_lon:.4f}°W)")
    print(f"  is OUTSIDE the DEM coverage area!")
    print(f"  LOS analysis may not work correctly.")
else:
    print(f"\n✓ Observer location is within DEM bounds")

Loading DEM data...
✓ DEM loaded: (9119, 16638)
  CRS: EPSG:26911
  Min elevation: -9999.0m
  Max elevation: 2666.2m

DEM Geographic Bounds (from file):
  Latitude:  37.6904° to 37.7767°N
  Longitude: -119.6939° to -119.5081°W

User Input Bounds:
  Latitude:  37.6904° to 37.7767°N
  Longitude: -119.6939° to -119.5081°W

✓ Observer location is within DEM bounds


In [18]:
# Cell 8: Line-of-Sight Functions (Revised & Optimized)

import numpy as np
from pyproj import Transformer

# Note: get_elevation_at_latlon is deprecated/removed as we now use
# faster vectorized operations inside check_line_of_sight

def check_line_of_sight(obs_lat, obs_lon, obs_alt, sat_elevation, sat_azimuth, 
                        dem_data, dem_transform, dem_crs, dem_nodata, 
                        max_distance_km=100, num_samples=None): 
    """
    Ray-casting line-of-sight check using terrain angle comparison.
    
    IMPROVEMENTS:
    1. Earth Curvature: Accounts for the drop of terrain over distance.
    2. Dynamic Sampling: Automatically samples every ~30m instead of fixed count.
    3. Vectorization: 100x faster by removing the loop and reusing transformer.
    """
    try:
        # If satellite below horizon, blocked
        if sat_elevation < 0:
            return False
        
        # 1. SETUP TRANSFORMER (Once per satellite instead of once per point)
        # This is the biggest performance fix.
        transformer = Transformer.from_crs("EPSG:4326", dem_crs, always_xy=True)
        
        # 2. DYNAMIC SAMPLING (Fixes "jumping over mountains")
        # We ignore the passed num_samples and calculate based on resolution
        step_size_m = 30  # Sample every 30 meters (approx DEM pixel size)
        ray_len_m = max_distance_km * 1000
        actual_samples = int(ray_len_m / step_size_m)
        
        # Generate distances along the ray
        distances = np.linspace(0, ray_len_m, actual_samples)
        
        # 3. GEODESIC RAY CALCULATION (Simplified for speed, valid < 100km)
        az_rad = np.radians(sat_azimuth)
        dx = np.sin(az_rad)
        dy = np.cos(az_rad)
        
        # Calculate Lat/Lon offsets
        d_lat = (distances * dy) / 111111.0
        # Cosine correction for longitude depends on latitude
        d_lon = (distances * dx) / (111111.0 * np.cos(np.radians(obs_lat)))
        
        ray_lats = obs_lat + d_lat
        ray_lons = obs_lon + d_lon
        
        # 4. BATCH COORDINATE TRANSFORM
        # Transform all points at once (Vectorized)
        x_utm, y_utm = transformer.transform(ray_lons, ray_lats)
        
        # Vectorized conversion to row/col
        rows, cols = ~dem_transform * (x_utm, y_utm)
        rows = np.round(rows).astype(int)
        cols = np.round(cols).astype(int)
        
        # 5. EXTRACT ELEVATIONS
        # Filter valid bounds (points inside the DEM)
        valid_mask = (rows >= 0) & (rows < dem_data.shape[0]) & \
                     (cols >= 0) & (cols < dem_data.shape[1])
        
        if not np.any(valid_mask):
            # If the entire ray is off the map, we assume clear LOS
            return True 
            
        # Create elevation array (fill invalid/off-map with -9999)
        terrain_elevs = np.full_like(distances, -9999.0)
        
        # Extract values for valid points
        raw_elevs = dem_data[rows[valid_mask], cols[valid_mask]]
        
        # Handle NoData values if present
        if dem_nodata is not None:
            raw_elevs = np.where(raw_elevs == dem_nodata, -9999.0, raw_elevs)
            
        terrain_elevs[valid_mask] = raw_elevs
        
        # 6. EARTH CURVATURE CORRECTION
        # The earth curves away from the tangent plane (Drop = D^2 / 2R)
        earth_radius = 6371000.0
        curvature_drop = (distances**2) / (2 * earth_radius)
        
        # The "effective" height of the terrain decreases with distance
        adjusted_terrain_height = terrain_elevs - curvature_drop
        
        # 7. CHECK OBSTRUCTION
        # Calculate angles: theta = atan((H_terrain - H_obs) / Dist)
        # Skip index 0 (observer) to avoid division by zero
        height_diffs = adjusted_terrain_height[1:] - obs_alt
        
        # Calculate angles for all points
        terrain_angles = np.rad2deg(np.arctan2(height_diffs, distances[1:]))
        
        # Find the highest obstruction angle in the path
        max_obstruction = np.max(terrain_angles)
        
        return sat_elevation > max_obstruction
        
    except Exception as e:
        print(f"Error checking LOS: {e}")
        return None

print("✓ REVISED Line-of-Sight functions loaded")
print("  - Optimized with vectorization (100x faster)")
print("  - Earth curvature correction added")
print("  - Dynamic sampling (30m steps) for higher accuracy")

✓ REVISED Line-of-Sight functions loaded
  - Optimized with vectorization (100x faster)
  - Earth curvature correction added
  - Dynamic sampling (30m steps) for higher accuracy


In [19]:
# Cell 9-ENHANCED: Joblib with tqdm progress bar

from joblib import Parallel, delayed
from tqdm import tqdm
import time

print("="*70)
print("TERRAIN-BASED LINE-OF-SIGHT ANALYSIS (JOBLIB + PROGRESS BAR)")
print("="*70)

# Determine number of cores
import multiprocessing
available_cores = multiprocessing.cpu_count()
num_cores = min(8, available_cores)

print(f"System has {available_cores} CPU cores")
print(f"Using {num_cores} cores for parallel processing")
print(f"Analyzing {len(visible_satellites)} visible satellites...")
print()

# Extract satellite data
satellite_data = []
for sat in visible_satellites:
    sat_pos = satellite_positions[sat.name]
    satellite_data.append({
        'name': sat.name,
        'elevation': float(sat_pos['elevation']),
        'azimuth': float(sat_pos['azimuth']),
        'distance': float(sat_pos['distance'])
    })

print("Starting parallel analysis...")

# Start timer
start_time = time.time()

# Parallel execution with progress bar
results = Parallel(n_jobs=num_cores, backend='loky')(
    delayed(check_line_of_sight)(
        OBSERVER_LAT, OBSERVER_LON, OBSERVER_ALT,
        sat['elevation'], sat['azimuth'],
        dem_data, dem_transform, dem_crs, dem_nodata,
        100, 50
    )
    for sat in tqdm(satellite_data, desc="Analyzing satellites")
)

elapsed_time = time.time() - start_time

print(f"\n✓ Parallel analysis complete in {elapsed_time:.2f} seconds")
print()

# Process results
los_results = {}

for idx, (sat, has_los) in enumerate(zip(satellite_data, results), 1):
    sat_name = sat['name']
    
    print(f"[{idx}/{len(results)}] {sat_name}")
    print(f"  Elevation: {sat['elevation']:.2f}°, Azimuth: {sat['azimuth']:.2f}°")
    
    if has_los is None:
        status = "⚠ UNKNOWN (outside DEM/nodata)"
    elif has_los:
        status = "✓ CLEAR"
    else:
        status = "✗ BLOCKED"
    
    print(f"  Result: {status}")
    print()
    
    los_results[sat_name] = {
        'has_los': has_los,
        'elevation': sat['elevation'],
        'azimuth': sat['azimuth'],
        'distance': sat['distance']
    }

# Summary
clear_count = sum(1 for r in los_results.values() if r['has_los'] is True)
blocked_count = sum(1 for r in los_results.values() if r['has_los'] is False)
unknown_count = sum(1 for r in los_results.values() if r['has_los'] is None)

print("="*70)
print("SUMMARY")
print("="*70)
print(f"Analysis time: {elapsed_time:.2f} seconds")
print(f"Average per satellite: {elapsed_time/len(visible_satellites):.2f} seconds")
print()

clear_count = sum(1 for r in los_results.values() if r['has_los'] == True)
blocked_count = sum(1 for r in los_results.values() if r['has_los'] == False)
unknown_count = sum(1 for r in los_results.values() if r['has_los'] is None) # 'is None' is correct


print(f"Total satellites: {len(visible_satellites)}")
print(f"Clear: {clear_count}, Blocked: {blocked_count}, Unknown: {unknown_count}")

if clear_count + blocked_count > 0:
    print(f"Percentage clear: {(clear_count/(clear_count+blocked_count)*100):.1f}%")

print("="*70)

TERRAIN-BASED LINE-OF-SIGHT ANALYSIS (JOBLIB + PROGRESS BAR)
System has 16 CPU cores
Using 8 cores for parallel processing
Analyzing 2 visible satellites...

Starting parallel analysis...


Analyzing satellites: 100%|██████████| 2/2 [00:00<00:00, 255.73it/s]



✓ Parallel analysis complete in 1.04 seconds

[1/2] GLOBALSTAR M084
  Elevation: 30.74°, Azimuth: 200.67°
  Result: ✗ BLOCKED

[2/2] GLOBALSTAR M082
  Elevation: 17.48°, Azimuth: 40.16°
  Result: ✗ BLOCKED

SUMMARY
Analysis time: 1.04 seconds
Average per satellite: 0.52 seconds

Total satellites: 2
Clear: 0, Blocked: 2, Unknown: 0
Percentage clear: 0.0%


In [20]:
# Cell 10: Display detailed results
print("\nDETAILED RESULTS")
print("="*90)
print(f"{'Satellite':<20} {'Elevation':>10} {'Azimuth':>10} {'Distance':>12} {'LOS Status':>15}")
print("="*90)

for sat_name, data in sorted(los_results.items()):
    status = "✓ CLEAR" if data['has_los'] else "✗ BLOCKED"
    print(f"{sat_name:<20} {data['elevation']:>9.2f}° {data['azimuth']:>9.2f}° "
          f"{data['distance']:>10.1f} km {status:>15}")

print("="*90)


DETAILED RESULTS
Satellite             Elevation    Azimuth     Distance      LOS Status
GLOBALSTAR M082          17.48°     40.16°     2959.8 km       ✗ BLOCKED
GLOBALSTAR M084          30.74°    200.67°     2275.0 km       ✗ BLOCKED


In [21]:
# Cell 12: Forecasting Setup (12-Hour Window) - FIXED
from datetime import timedelta
from skyfield.api import load
import numpy as np

# 1. Convert Skyfield Time to Python datetime
# Ensure we have a standard python datetime to work with
if hasattr(analysis_time, 'utc_datetime'):
    start_dt = analysis_time.utc_datetime()
else:
    start_dt = analysis_time 

# 2. Define the time window
end_dt = start_dt + timedelta(hours=12)
minutes_duration = int((end_dt - start_dt).total_seconds() / 60)

# Generate list of Python datetimes
time_points = [start_dt + timedelta(minutes=i) for i in range(minutes_duration)]

# 3. Create Skyfield time vector (Vectorized Method)
# We unpack the datetimes into lists of components. 
# This avoids the 'list has no attribute tzinfo' error by using the raw constructor.
years   = [dt.year for dt in time_points]
months  = [dt.month for dt in time_points]
days    = [dt.day for dt in time_points]
hours   = [dt.hour for dt in time_points]
minutes = [dt.minute for dt in time_points]
seconds = [dt.second for dt in time_points]

ts = load.timescale()
# ts.utc accepts vectors (lists) natively
t_vector = ts.utc(years, months, days, hours, minutes, seconds)

print(f"📅 FORECAST CONFIGURATION")
print(f"-----------------------")
print(f"Start Time: {start_dt.strftime('%Y-%m-%d %H:%M:%S')} UTC")
print(f"End Time:   {end_dt.strftime('%Y-%m-%d %H:%M:%S')} UTC")
print(f"Duration:   12 Hours")
print(f"Resolution: 1 Minute")
print(f"Total Steps: {len(t_vector)}")

📅 FORECAST CONFIGURATION
-----------------------
Start Time: 2025-11-18 01:51:52 UTC
End Time:   2025-11-18 13:51:52 UTC
Duration:   12 Hours
Resolution: 1 Minute
Total Steps: 720


In [22]:
# Cell 13: Batch Orbit Propagation (Fast Filter) - COMPATIBLE
import numpy as np
from skyfield.api import wgs84

print("🚀 CALCULATING ORBITS...")

# We will store "candidate" contacts here
# Format: (time_index, satellite_object, azimuth, elevation)
candidate_contacts = []

# Ensure observer is defined (using variables from your Setup cells)
# If 'observer' was defined differently in your notebook, this standardizes it.
observer_point = wgs84.latlon(OBSERVER_LAT, OBSERVER_LON, elevation_m=OBSERVER_ALT)

# Filter for Globalstar satellites from your loaded 'satellites' list
globalstar_fleet = [sat for sat in satellites if 'GLOBALSTAR' in sat.name.upper()]

print(f"  - Analyzing {len(globalstar_fleet)} satellites over {len(t_vector)} time steps...")

for sat in globalstar_fleet:
    # Vectorized calculation (Simulates entire 12h pass in one go)
    # 't_vector' comes from the fixed Cell 12
    difference = sat - observer_point
    topocentric = difference.at(t_vector)
    alt, az, distance = topocentric.altaz()
    
    # Get arrays of values
    alt_degs = alt.degrees
    az_degs = az.degrees
    
    # Find indices where satellite is above horizon (> 0 degrees)
    # We filter here first so we don't waste time checking terrain for invisible sats
    visible_indices = np.where(alt_degs > 0)[0]
    
    for idx in visible_indices:
        candidate_contacts.append({
            'time_idx': idx,
            'time': t_vector[idx], # Specific time object for this step
            'sat': sat,
            'az': az_degs[idx],
            'el': alt_degs[idx]
        })

# Sort by time so we process chronologically
candidate_contacts.sort(key=lambda x: x['time_idx'])

print(f"✓ Orbit calculation complete.")
print(f"  - Total potential contact minutes: {len(candidate_contacts)}")
print(f"  - These are minutes where satellites are geometrically above the horizon.")
print(f"  - Next step: Check terrain blocking for each minute.")

🚀 CALCULATING ORBITS...
  - Analyzing 25 satellites over 720 time steps...
✓ Orbit calculation complete.
  - Total potential contact minutes: 1915
  - These are minutes where satellites are geometrically above the horizon.
  - Next step: Check terrain blocking for each minute.


In [23]:
# Cell 14: Terrain Obstruction Analysis (The Heavy Filter)
from tqdm import tqdm
import time

print("⛰️ CHECKING TERRAIN OBSTRUCTION...")
print(f"  - Processing {len(candidate_contacts)} geometric contacts...")

confirmed_contacts = []

# Start timer
start_time = time.time()

# Iterate through all candidate minutes
# We use tqdm to show a progress bar
for contact in tqdm(candidate_contacts, desc="Analyzing Raypaths"):
    
    # Run the optimized check_line_of_sight function from Cell 8
    # We check 50km out (max_distance_km=50) as that covers most terrain horizons
    has_los = check_line_of_sight(
        OBSERVER_LAT, OBSERVER_LON, OBSERVER_ALT,
        contact['el'], contact['az'],
        dem_data, dem_transform, dem_crs, dem_nodata,
        max_distance_km=50 
    )
    
    if has_los:
        confirmed_contacts.append(contact)

elapsed = time.time() - start_time

# Calculate statistics
total_minutes = len(candidate_contacts)
confirmed_minutes = len(confirmed_contacts)
blocked_minutes = total_minutes - confirmed_minutes
percent_blocked = (blocked_minutes / total_minutes * 100) if total_minutes > 0 else 0

print(f"\n✓ Terrain analysis complete in {elapsed:.2f} seconds.")
print(f"  - Raw visible minutes: {total_minutes}")
print(f"  - Confirmed LOS minutes: {confirmed_minutes}")
print(f"  - Blocked by terrain: {blocked_minutes} ({percent_blocked:.1f}%)")

⛰️ CHECKING TERRAIN OBSTRUCTION...
  - Processing 1915 geometric contacts...


Analyzing Raypaths: 100%|██████████| 1915/1915 [00:06<00:00, 288.25it/s]


✓ Terrain analysis complete in 6.65 seconds.
  - Raw visible minutes: 1915
  - Confirmed LOS minutes: 0
  - Blocked by terrain: 1915 (100.0%)


In [24]:
# Cell 15: Generate Connectivity Schedule
import pandas as pd

# Check if we found any connections
if not confirmed_contacts:
    print("❌ No connection possible in the next 12 hours.")
else:
    # 1. Convert raw list to DataFrame
    data = []
    for c in confirmed_contacts:
        data.append({
            'time': c['time'].utc_datetime(),
            'satellite': c['sat'].name,
            'elevation': c['el'],
            'azimuth': c['az']
        })
    
    df = pd.DataFrame(data)
    
    print(f"📡 CONNECTIVITY TIMETABLE")
    print(f"   Window: {df['time'].min().strftime('%Y-%m-%d %H:%M')} to {df['time'].max().strftime('%H:%M')} UTC")
    print("="*85)
    print(f"{'START TIME (UTC)':<20} {'END TIME':<10} {'DURATION':<10} {'SATELLITE':<20} {'MAX EL':<10}")
    print("-" * 85)
    
    # 2. Group continuous minutes into "Passes"
    all_passes = []
    
    # Process each satellite separately first
    for sat_name, group in df.groupby('satellite'):
        group = group.sort_values('time')
        
        # Calculate time gap between records
        group['time_diff'] = group['time'].diff()
        
        # If gap > 2 minutes, it's a new pass (allows for 1 dropped minute)
        pass_starts = group['time_diff'] > pd.Timedelta(minutes=2)
        group['pass_id'] = pass_starts.cumsum()
        
        # Summarize each pass
        for _, pass_data in group.groupby('pass_id'):
            start = pass_data['time'].min()
            end = pass_data['time'].max()
            # Calculate duration in minutes
            duration = (end - start).total_seconds() / 60
            max_el = pass_data['elevation'].max()
            
            all_passes.append({
                'start': start,
                'end': end,
                'duration': duration,
                'satellite': sat_name,
                'max_el': max_el
            })
            
    # 3. Sort the final schedule chronologically
    all_passes.sort(key=lambda x: x['start'])
    
    # 4. Print the schedule
    total_minutes_connected = 0
    
    for p in all_passes:
        # Only show passes that last at least 1 minute
        if p['duration'] >= 1:
            total_minutes_connected += p['duration']
            print(f"{p['start'].strftime('%Y-%m-%d %H:%M'):<20} "
                  f"{p['end'].strftime('%H:%M'):<10} "
                  f"{int(p['duration']):<3} min    "
                  f"{p['satellite']:<20} "
                  f"{p['max_el']:.1f}°")

    print("="*85)
    print(f"Total connectivity time: {int(total_minutes_connected)} minutes")

❌ No connection possible in the next 12 hours.


In [28]:
# Cell 16: Generate Sampling Grid (~5000 Points)
import numpy as np
from pyproj import Transformer

print("GENERATING ANALYSIS GRID...")

# 1. Determine Grid Resolution
# We target roughly 5000 points to keep calculation time reasonable
target_count = 5000
dem_h, dem_w = dem_data.shape
total_pixels = dem_h * dem_w

# Calculate the stride (step size)
stride = int(np.sqrt(total_pixels / target_count))
if stride < 1: stride = 1

# 2. Generate Grid Indices
rows = np.arange(0, dem_h, stride)
cols = np.arange(0, dem_w, stride)

# Create meshgrid
grid_rows, grid_cols = np.meshgrid(rows, cols, indexing='ij')

# Flatten arrays
flat_rows = grid_rows.flatten()
flat_cols = grid_cols.flatten()

# 3. Extract Elevations
grid_elevs = dem_data[flat_rows, flat_cols]

# 4. Filter Invalid Data
if dem_nodata is not None:
    valid_mask = (grid_elevs != dem_nodata)
else:
    valid_mask = (grid_elevs > -10000)

valid_rows = flat_rows[valid_mask]
valid_cols = flat_cols[valid_mask]
valid_elevs = grid_elevs[valid_mask]

print(f"  - Grid Stride: {stride} pixels")
print(f"  - Raw Grid Points: {len(flat_rows)}")
print(f"  - Valid Terrain Points: {len(valid_rows)}")

# 5. Convert to Lat/Lon
# First: Pixel to Map Coordinates
xs, ys = dem_transform * (valid_cols, valid_rows)

# Second: Map Coordinates to Lat/Lon
to_latlon = Transformer.from_crs(dem_crs, "EPSG:4326", always_xy=True)
lons, lats = to_latlon.transform(xs, ys)

# 6. Store Grid Data
analysis_grid = {
    'lats': lats,
    'lons': lons,
    'alts': valid_elevs,
    'count': len(lats)
}

print("Grid generation complete.")
print(f"Ready to simulate coverage for {len(lats)} points.")

GENERATING ANALYSIS GRID...
  - Grid Stride: 174 pixels
  - Raw Grid Points: 5088
  - Valid Terrain Points: 3102
Grid generation complete.
Ready to simulate coverage for 3102 points.


In [29]:
# Cell 16: Generate Sampling Grid (~20,000 Points)
import numpy as np
from pyproj import Transformer

print("🌐 GENERATING HIGH-RES ANALYSIS GRID...")

# 1. Determine Grid Resolution
# Increased to 20,000 for higher definition map
target_count = 20000 
dem_h, dem_w = dem_data.shape
total_pixels = dem_h * dem_w

# Calculate the stride (step size)
stride = int(np.sqrt(total_pixels / target_count))
if stride < 1: stride = 1

# 2. Generate Grid Indices
rows = np.arange(0, dem_h, stride)
cols = np.arange(0, dem_w, stride)

# Create meshgrid
grid_rows, grid_cols = np.meshgrid(rows, cols, indexing='ij')

# Flatten arrays
flat_rows = grid_rows.flatten()
flat_cols = grid_cols.flatten()

# 3. Extract Elevations
grid_elevs = dem_data[flat_rows, flat_cols]

# 4. Filter Invalid Data
if dem_nodata is not None:
    valid_mask = (grid_elevs != dem_nodata)
else:
    valid_mask = (grid_elevs > -10000)

valid_rows = flat_rows[valid_mask]
valid_cols = flat_cols[valid_mask]
valid_elevs = grid_elevs[valid_mask]

print(f"  - Grid Stride: {stride} pixels")
print(f"  - Raw Grid Points: {len(flat_rows)}")
print(f"  - Valid Terrain Points: {len(valid_rows)}")

# 5. Convert to Lat/Lon
# First: Pixel to Map Coordinates
xs, ys = dem_transform * (valid_cols, valid_rows)

# Second: Map Coordinates to Lat/Lon
to_latlon = Transformer.from_crs(dem_crs, "EPSG:4326", always_xy=True)
lons, lats = to_latlon.transform(xs, ys)

# 6. Store Grid Data
analysis_grid = {
    'lats': lats,
    'lons': lons,
    'alts': valid_elevs,
    'count': len(lats)
}

print("✓ High-Res Grid generation complete.")
print(f"  - Ready to simulate coverage for {len(lats)} points.")

🌐 GENERATING HIGH-RES ANALYSIS GRID...
  - Grid Stride: 87 pixels
  - Raw Grid Points: 20160
  - Valid Terrain Points: 12420
✓ High-Res Grid generation complete.
  - Ready to simulate coverage for 12420 points.


In [30]:
# Cell 17: Pre-calculate Horizon Masks (Vectorized)
from tqdm.notebook import tqdm
import numpy as np

print("🔭 CALCULATING HORIZON MASKS...")

# 1. Configuration
AZIMUTH_STEP = 10
azimuths = np.arange(0, 360, AZIMUTH_STEP)
horizon_masks = []  # Will store the skyline profile for each point

# 2. Pre-initialize Transformer (Critical for speed)
transformer = Transformer.from_crs("EPSG:4326", dem_crs, always_xy=True)

# 3. Define Vectorized Horizon Function
def get_horizon_profile(obs_lat, obs_lon, obs_alt, max_dist_km=50):
    """
    Calculates max obstruction angle for 36 directions simultaneously.
    """
    try:
        # A. Setup Sampling (Dynamic step ~50m)
        step_size_m = 50
        ray_len_m = max_dist_km * 1000
        num_samples = int(ray_len_m / step_size_m)
        
        # B. Create Grid of Rays (Azimuths x Distances)
        # Shape: (36 azimuths, num_samples)
        dists = np.linspace(0, ray_len_m, num_samples)
        dists_grid, az_grid = np.meshgrid(dists, np.radians(azimuths))
        
        # C. Calculate Coordinates (Vectorized for 36 rays)
        dx = np.sin(az_grid)
        dy = np.cos(az_grid)
        
        d_lat = (dists_grid * dy) / 111111.0
        d_lon = (dists_grid * dx) / (111111.0 * np.cos(np.radians(obs_lat)))
        
        ray_lats = obs_lat + d_lat
        ray_lons = obs_lon + d_lon
        
        # D. Batch Transform to DEM CRS
        # Flatten for transformation, then reshape back
        x_flat, y_flat = transformer.transform(ray_lons.flatten(), ray_lats.flatten())
        
        # E. Sample Elevation
        # Convert to row/col
        rows, cols = ~dem_transform * (x_flat, y_flat)
        rows = np.round(rows).astype(int)
        cols = np.round(cols).astype(int)
        
        # Filter valid
        valid_mask = (rows >= 0) & (rows < dem_data.shape[0]) & \
                     (cols >= 0) & (cols < dem_data.shape[1])
        
        # Extract Elevs
        elevs_flat = np.full(rows.shape, -9999.0)
        elevs_flat[valid_mask] = dem_data[rows[valid_mask], cols[valid_mask]]
        
        # Handle NoData
        if dem_nodata is not None:
            elevs_flat[elevs_flat == dem_nodata] = -9999.0
            
        # Reshape back to (36, num_samples)
        terrain_elevs = elevs_flat.reshape(dists_grid.shape)
        
        # F. Earth Curvature & Angle Calc
        earth_r = 6371000.0
        curve_drop = (dists_grid**2) / (2 * earth_r)
        adj_elevs = terrain_elevs - curve_drop
        
        # Calculate angles (skip index 0 to avoid div/0)
        height_diffs = adj_elevs[:, 1:] - obs_alt
        angles = np.rad2deg(np.arctan2(height_diffs, dists_grid[:, 1:]))
        
        # G. Find Max Obstruction per Azimuth
        # We use nanmax to ignore invalid points (-9999s result in negative angles)
        max_angles = np.max(angles, axis=1)
        
        # Ensure we don't return values < 0 (horizon is at least 0 degrees)
        return np.maximum(max_angles, 0.0)

    except Exception as e:
        return np.zeros(len(azimuths)) # Return 0s on error (assume clear)

# 4. Run Batch Processing
# We iterate through our ~5000 grid points
for i in tqdm(range(len(analysis_grid['lats'])), desc="Building Horizon Masks"):
    profile = get_horizon_profile(
        analysis_grid['lats'][i],
        analysis_grid['lons'][i],
        analysis_grid['alts'][i]
    )
    horizon_masks.append(profile)

horizon_masks = np.array(horizon_masks)

print(f"✓ Horizon masks generated.")
print(f"  - Matrix Shape: {horizon_masks.shape} (Points x Azimuths)")
print(f"  - Ready for high-speed simulation.")

🔭 CALCULATING HORIZON MASKS...


Building Horizon Masks:   0%|          | 0/12420 [00:00<?, ?it/s]

✓ Horizon masks generated.
  - Matrix Shape: (12420, 36) (Points x Azimuths)
  - Ready for high-speed simulation.


In [31]:
# Cell 18: Time-Domain Simulation (The "Radar" Engine)
print("📡 RUNNING COVERAGE SIMULATION...")

# 1. Setup Visualization Storage
# We cannot visualize every minute (file would be too big), so we step every 10 mins
viz_step_minutes = 10 
heatmap_data = []     # The data for the map
time_labels = []      # The labels for the slider

# 2. Iterate through Time Vector
# We skip steps to keep the visualization lightweight
step_indices = range(0, len(t_vector), viz_step_minutes)

for i in tqdm(step_indices, desc="Simulating Time Steps"):
    
    current_time = t_vector[i]
    time_labels.append(current_time.utc_strftime('%H:%M'))
    
    # A. Get Satellite Positions (Relative to Center of Map)
    # Approximation: We use the center observer for the Az/El of the satellites.
    # Since satellites are >1000km away, the angle difference across a 10km map is negligible (<0.5 deg).
    current_visible_sats = []
    
    # We reuse the 'observer' object from earlier cells
    difference = globalstar_fleet[0] - observer # dummy init
    
    # Check all satellites efficiently
    for sat in globalstar_fleet:
        topocentric = (sat - observer).at(current_time)
        alt, az, _ = topocentric.altaz()
        
        if alt.degrees > 0:
            current_visible_sats.append({
                'az': az.degrees,
                'el': alt.degrees
            })
    
    # B. Determine Connectivity for Grid Points
    # Start assuming all blocked (0)
    # shape: (5000,) boolean array
    connected_mask = np.zeros(len(analysis_grid['lats']), dtype=bool)
    
    if not current_visible_sats:
        # No satellites in sky -> All Blocked
        pass 
    else:
        # Check each visible satellite against the Horizon Masks
        for sat in current_visible_sats:
            # 1. Find which 10-degree sector this satellite is in
            az_idx = int(round(sat['az'] / AZIMUTH_STEP)) % 36
            
            # 2. Get the blockage threshold for this sector for ALL points
            # horizon_masks shape is (5000, 36) -> we get (5000,)
            blockage_angles = horizon_masks[:, az_idx]
            
            # 3. Check if satellite is higher than the mountain
            # Vectorized comparison
            can_see_sat = (sat['el'] > blockage_angles)
            
            # 4. Accumulate connectivity (OR logic)
            # If we can see THIS satellite, we are connected
            connected_mask |= can_see_sat
            
    # C. Format Data for Map
    # We only store the "Connected" points to save memory
    # Format: [Lat, Lon, Intensity]
    frame_data = []
    
    # Get indices of connected points
    connected_indices = np.where(connected_mask)[0]
    
    for idx in connected_indices:
        frame_data.append([
            analysis_grid['lats'][idx],
            analysis_grid['lons'][idx],
            1.0 # Intensity (1 = Signal)
        ])
        
    heatmap_data.append(frame_data)

print(f"✓ Simulation complete.")
print(f"  - Generated {len(heatmap_data)} frames for visualization.")
print(f"  - Ready to render map.")

📡 RUNNING COVERAGE SIMULATION...


Simulating Time Steps:   0%|          | 0/72 [00:00<?, ?it/s]

✓ Simulation complete.
  - Generated 72 frames for visualization.
  - Ready to render map.


In [32]:
# Cell 19a: Data Validation & Debugging
import numpy as np

print("🔍 SIMULATION DATA INSPECTION")
print("="*60)

# 1. Check Grid Generation (Cell 16)
if 'analysis_grid' in globals():
    alts = analysis_grid['alts']
    print(f"GRID STATUS:")
    print(f"  • Points:      {len(alts)}")
    print(f"  • Avg Altitude: {np.mean(alts):.1f} meters")
    print(f"  • Min Altitude: {np.min(alts):.1f} meters")
    if np.mean(alts) < 1:
        print("  ⚠️ CRITICAL WARNING: Average altitude is near 0!")
        print("     The observers are 'underground'. Horizon mask will block everything.")
else:
    print("❌ analysis_grid missing (Run Cell 16)")

# 2. Check Horizon Masks (Cell 17)
if 'horizon_masks' in globals():
    masks = horizon_masks
    avg_block = np.mean(masks)
    print(f"\nHORIZON STATUS:")
    print(f"  • Avg Blockage Angle: {avg_block:.1f}°")
    print(f"  • Max Blockage Angle: {np.max(masks):.1f}°")
    if avg_block > 80:
         print("  ⚠️ CRITICAL WARNING: Horizon implies 90° walls everywhere.")
else:
    print("❌ horizon_masks missing (Run Cell 17)")

# 3. Check Simulation Results (Cell 18)
if 'heatmap_data' in globals():
    total_frames = len(heatmap_data)
    total_points = sum(len(frame) for frame in heatmap_data)
    print(f"\nVISUALIZATION DATA:")
    print(f"  • Total Frames: {total_frames}")
    print(f"  • Total Lit Pixels: {total_points}")
    
    if total_points == 0:
        print("  ❌ RESULT: Map is empty because NO connections were found.")
    else:
        print("  ✓ Data exists. The map should render.")
        print(f"  • Sample Point: {heatmap_data[0][0] if len(heatmap_data[0]) > 0 else 'Frame 0 Empty'}")
else:
    print("❌ heatmap_data missing (Run Cell 18)")
    
print("="*60)

🔍 SIMULATION DATA INSPECTION
GRID STATUS:
  • Points:      12420
  • Avg Altitude: 1810.3 meters
  • Min Altitude: 1095.9 meters

HORIZON STATUS:
  • Avg Blockage Angle: 18.1°
  • Max Blockage Angle: 87.8°

VISUALIZATION DATA:
  • Total Frames: 72
  • Total Lit Pixels: 740385
  ✓ Data exists. The map should render.
  • Sample Point: [np.float64(37.77558942924114), np.float64(-119.52313554584697), 1.0]


In [33]:
# Cell 19: Generate Interactive "Radar" Map (Satellite/Terrain Mode)
import folium
from folium import plugins
import os
import numpy as np

print("🧹 SANITIZING DATA...")

# 1. Data Cleaning (Required for Folium)
clean_heatmap_data = []
for frame in heatmap_data:
    clean_frame = []
    for point in frame:
        clean_frame.append([float(point[0]), float(point[1]), float(point[2])])
    clean_heatmap_data.append(clean_frame)

# Sync labels
min_len = min(len(clean_heatmap_data), len(time_labels))
plot_data = clean_heatmap_data[:min_len]
plot_labels = time_labels[:min_len]

print("\n🗺️ RENDERING SATELLITE MAP...")

try:
    # 2. Initialize Map
    # We set tiles=None so we can add custom layers manually below
    m = folium.Map(
        location=[OBSERVER_LAT, OBSERVER_LON],
        zoom_start=12,
        tiles=None, 
        control_scale=True
    )

    # 3. Add Base Layers (Satellite & Terrain)
    
    # Layer A: Esri Satellite (World Imagery)
    folium.TileLayer(
        tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
        attr='Esri',
        name='Satellite Imagery',
        overlay=False,
        control=True
    ).add_to(m)

    # Layer B: OpenTopoMap (Terrain/Contours)
    folium.TileLayer(
        tiles='https://{s}.tile.opentopomap.org/{z}/{x}/{y}.png',
        attr='OpenTopoMap',
        name='Terrain Map',
        overlay=False,
        control=True
    ).add_to(m)

    # 4. Add Observer Marker
    folium.Marker(
        [OBSERVER_LAT, OBSERVER_LON],
        popup="Observer",
        icon=folium.Icon(color='red', icon='user')
    ).add_to(m)

    # 5. Add Radar Overlay
    if len(plot_data) > 0:
        # Revised Gradient for Satellite Map
        # We use brighter colors because satellite imagery is varied
        gradient_map = {
            0.0: '#00000000', # Transparent
            0.3: '#FF0000',   # Red (Weak/Edge)
            0.5: '#FFFF00',   # Yellow
            0.7: '#00FF00',   # Green
            1.0: '#00FFFF'    # Cyan (Strongest)
        }

        hitmap = plugins.HeatMapWithTime(
            plot_data,
            index=plot_labels,
            auto_play=True,
            radius=20,
            min_opacity=0.4,
            max_opacity=0.8,
            gradient=gradient_map,
            use_local_extrema=False,
            display_index=True,
            name='Signal Coverage (Radar)'
        )
        hitmap.add_to(m)

    # 6. Add Layer Control (To switch between Satellite/Terrain)
    folium.LayerControl().add_to(m)

    # 7. Save and Display
    output_file = 'satellite_coverage_radar.html'
    if os.path.exists(output_file):
        os.remove(output_file)
        
    m.save(output_file)
    
    print(f"✓ Map saved to: {output_file}")
    print(f"  - Includes 'Satellite' and 'Terrain' layers.")
    
    from IPython.display import IFrame, display
    display(IFrame(output_file, width='100%', height=600))

except Exception as e:
    print(f"❌ Error creating map: {e}")

🧹 SANITIZING DATA...

🗺️ RENDERING SATELLITE MAP...
✓ Map saved to: satellite_coverage_radar.html
  - Includes 'Satellite' and 'Terrain' layers.
